In [ ]:
import os, sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from nbafuns import *
from nba_api.stats.endpoints import leaguedashplayerptshot, playerdashptshots, leaguedashteamptshot

data_DIR = "../data/shots/"

In [ ]:
closest_def = ['0-2 Feet - Very Tight', '2-4 Feet - Tight', '4-6 Feet - Open', '6+ Feet - Wide Open']
general_range = ['Catch and Shoot', 'Pullups', 'Less Than 10 ft']
league, league_id = "NBA", "00"

In [ ]:
season_start = 2013
season_end = 2025
seasons = np.arange(season_start, season_end, 1)
for season in tqdm(seasons):
    dfa = []
    for a in general_range:
        stats = leaguedashplayerptshot.LeagueDashPlayerPtShot(
            league_id=league_id,
            season=get_ss(season),
            season_type_all_star="Regular Season",
            general_range_nullable = a,
        ).get_data_frames()
        df1 = stats[0]
        df1["general_range"] = a
        df1["season"] = season + 1 
        dfa.append(df1)
        time.sleep(0.6)
    with pd.option_context("future.no_silent_downcasting", True):                    
        dfa1 = [df2.fillna(0).infer_objects(copy=False) for df2 in dfa if not df2.empty]
    df2 = pd.concat(dfa1)
    df2.columns = map(str.lower,df2.columns)
    df2 = df2.reset_index(drop=True)
    df2.to_parquet(data_DIR + f"{league}_Shots_General_Range_{season}.parquet")

In [ ]:
season_start = 2013
season_end = 2025
seasons = np.arange(season_start, season_end, 1)
for season in tqdm(seasons):
    dfa = []
    for a in general_range:
        stats = leaguedashplayerptshot.LeagueDashPlayerPtShot(
            league_id=league_id,
            season=get_ss(season),
            season_type_all_star="Playoffs",
            general_range_nullable = a,
        ).get_data_frames()
        df1 = stats[0]
        df1["general_range"] = a
        df1["season"] = season + 1 
        dfa.append(df1)
        time.sleep(0.6)
    with pd.option_context("future.no_silent_downcasting", True):                    
        dfa1 = [df2.fillna(0).infer_objects(copy=False) for df2 in dfa if not df2.empty]
    df2 = pd.concat(dfa1)
    df2.columns = map(str.lower,df2.columns)
    df2 = df2.reset_index(drop=True)
    df2.to_parquet(data_DIR + f"{league}_Shots_General_Range_{season}_PS.parquet")

In [ ]:
dfa = []
dfb = []
for season in seasons:
    df1a = pd.read_parquet(data_DIR + f"{league}_Shots_General_Range_{season}.parquet")
    df1b = pd.read_parquet(data_DIR + f"{league}_Shots_General_Range_{season}_PS.parquet")
    dfa.append(df1a)
    dfb.append(df1b)

In [ ]:
df2a = pd.concat(dfa)
df2b = pd.concat(dfb)
df2a["Segment"] = "Reg Season"
df2b["Segment"] = "Playoffs"

In [ ]:
df3a = df2a.query("general_range == 'Pullups'")
df3b = df2b.query("general_range == 'Pullups'")

In [ ]:
df3 = pd.concat([df3a,df3b])

In [ ]:
pn = 'LeBron James'
df4 = df3.query(f"player_name == '{pn}'")
cols = ["season","fg3a","fg3_pct","Segment"]
df4 = df4[cols]
avg = df3["fg3m"].sum()/df3["fg3a"].sum()
avg

In [ ]:
# df4 = df4[df4["season"] != 2020]
p = (
    ggplot(df4)
    + aes(x="season",y="fg3_pct",color="Segment",group="Segment")
    + geom_point(size=2)
    + geom_line(size=1.2)
    + geom_hline(yintercept=avg, color = "black", linetype="dashed")
    + scale_y_continuous(labels=percent_format())
    + theme_idv
    + labs(
        title=f"{pn} Pull Up 3PT%",
        subtitle="Avg Pull Up 3PT% = 32.86% incl. both reg season and playoffs",
        x = "Season",
        y = "3 PT %"
    )
    + theme(
        plot_subtitle=element_text(size=12)
    )
    + pnba
)
p